In [ ]:
from google.colab import files
import os

uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

# Create a folder in your Google Drive
os.makedirs('/content/drive/your model path from drive', exist_ok=True)

# Copy all model files to Drive
files_to_copy = [
    'config.json',
    'label_map.json',
    'model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json'
]

for file in files_to_copy:
    shutil.copy(file, '/content/drive/your model path from drive')
    print(f"✅ Copied: {file}")

print("\n🎉 All files saved to Google Drive!")

Run from here!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load directly from Drive
import json
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

model_path = '/content/drive/your model path from drive'

with open(f"{model_path}/label_map.json", "r") as f:
    label_map = json.load(f)

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_path)

model.eval()
print("✅ Model loaded from Google Drive!")

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model loaded from Google Drive!


In [ ]:
def predict_priority(complaint_text):
    """Takes a complaint and returns predicted priority label."""

    inputs = tokenizer(
        complaint_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_index = str(torch.argmax(outputs.logits).item())
    priority = label_map[predicted_index]

    # Get confidence score (how sure the model is)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
    confidence = round(torch.max(probabilities).item() * 100, 2)

    return priority, confidence

In [ ]:
print("=" * 65)
print("             COMPLAINT PRIORITY PREDICTION RESULTS")
print("=" * 65)

test_complaints = [
("My internet has been down for 3 days, I work from home","High"),
("There is a gas leak in my building, this is an emergency","High"),
("I have been waiting for a refund for 2 weeks","Medium"),
("The app crashes sometimes when I open it", "Medium"),
("I want to update my billing address","Low"),
("Can I get a copy of last month's invoice please","Low"),
("the portal is down and i cant submit my assignment deadline is today","High"),
("a student was attacked in the hostel and security did nothing","High"),
("teacher hasnt returned our assignments for over a week","Medium"),
("the library has been closed for 4 days without any notice","Medium"),
("the cafeteria menu should have more variety","Low"),
("library should stay open later during exams","Low"),

]

correct = 0

for complaint, expected in test_complaints:
    predicted, confidence = predict_priority(complaint)
    status = "✅ Correct" if predicted == expected else "❌ Wrong"
    if predicted == expected:
        correct += 1

    print(f"\nComplaint  : {complaint}")
    print(f"Expected   : {expected}")
    print(f"Predicted  : {predicted}  (Confidence: {confidence}%)")
    print(f"Status     : {status}")
    print("-" * 65)

accuracy = round((correct / len(test_complaints)) * 100, 2)
print(f"\n📊 Quick Accuracy: {correct}/{len(test_complaints)} correct = {accuracy}%")

             COMPLAINT PRIORITY PREDICTION RESULTS

Complaint  : My internet has been down for 3 days, I work from home
Expected   : High
Predicted  : Medium  (Confidence: 96.29%)
Status     : ❌ Wrong
-----------------------------------------------------------------

Complaint  : There is a gas leak in my building, this is an emergency
Expected   : High
Predicted  : High  (Confidence: 99.98%)
Status     : ✅ Correct
-----------------------------------------------------------------

Complaint  : I have been waiting for a refund for 2 weeks
Expected   : Medium
Predicted  : Medium  (Confidence: 98.35%)
Status     : ✅ Correct
-----------------------------------------------------------------

Complaint  : The app crashes sometimes when I open it
Expected   : Medium
Predicted  : Low  (Confidence: 84.24%)
Status     : ❌ Wrong
-----------------------------------------------------------------

Complaint  : I want to update my billing address
Expected   : Low
Predicted  : Low  (Confidence: 99.61%

In [ ]:
# Let's see how confident the model is on each class
test_text = "There is a gas leak in girls hostel!"

inputs = tokenizer(test_text, return_tensors="pt", truncation=True, padding=True)
with torch.no_grad():
    outputs = model(**inputs)

probs = torch.nn.functional.softmax(outputs.logits, dim=1)[0]
print("Probability for each class:")
for idx, prob in enumerate(probs):
    print(f"  {label_map[str(idx)]}: {round(prob.item() * 100, 2)}%")

Probability for each class:
  High: 99.98%
  Low: 0.01%
  Medium: 0.01%
